In [1]:
# Clone KokoClone repository
!git clone https://github.com/Ashish-Patnaik/kokoclone.git
%cd kokoclone

# Install dependencies (GPU optimized for T4)
!pip install -r requirements.txt
!pip install kokoro-onnx[gpu]
!pip install datasets huggingface_hub scikit-learn soundfile

Cloning into 'kokoclone'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 57 (delta 27), reused 35 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 38.02 KiB | 6.34 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/kokoclone
  Cloning https://github.com/frothywater/kanade-tokenizer to /tmp/pip-req-build-1ruayh9s
  Running command git clone --filter=blob:none --quiet https://github.com/frothywater/kanade-tokenizer /tmp/pip-req-build-1ruayh9s
  Resolved https://github.com/frothywater/kanade-tokenizer to commit 8e17729e9b96509606a01d4be959aa9c67d6a92a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.8 MB/s eta 0:00:00
  Installing build d

In [2]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from datasets import load_dataset
import soundfile as sf
import numpy as np

print("Connecting to ai4bharat/indicvoices_r...")

# Load dataset in streaming mode
ds = load_dataset("ai4bharat/indicvoices_r", "Hindi", split="train", streaming=True)

ref_male_id = "S4257210000350170"
ref_female_id = "S4259138800367594"

male_audio_saved = False
female_audio_saved = False

# Search and extract specific speaker audio
for sample in ds:
    # Safely extract speaker ID if nested or directly accessible
    current_speaker = sample.get("speaker_id")

    if current_speaker == ref_male_id and not male_audio_saved:
        audio_data = sample["audio"]
        sf.write("ref_male.wav", audio_data["array"], audio_data["sampling_rate"])
        print(f"Saved Male Reference Voice: {ref_male_id}")
        male_audio_saved = True

    if current_speaker == ref_female_id and not female_audio_saved:
        audio_data = sample["audio"]
        sf.write("ref_female.wav", audio_data["array"], audio_data["sampling_rate"])
        print(f"Saved Female Reference Voice: {ref_female_id}")
        female_audio_saved = True

    if male_audio_saved and female_audio_saved:
        break

print("Reference audios successfully extracted.")

Connecting to ai4bharat/indicvoices_r...


README.md:   0%|          | 0.00/33.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/99 [00:00<?, ?it/s]

Saved Female Reference Voice: S4259138800367594
Saved Male Reference Voice: S4257210000350170
Reference audios successfully extracted.


In [5]:
import json
import os
from sklearn.model_selection import train_test_split

# Ensure the path points to where you uploaded the JSON file
dataset_path = '/content/hindi_evaluation_set.json'

with open(dataset_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Convert dictionary to a list of tuples for splitting
items = list(data.items())

# Split the dataset: 80% train, 20% unseen test
train_items, test_items = train_test_split(items, test_size=0.2, random_state=42)

# Convert back to dictionaries
train_data = dict(train_items)
test_data = dict(test_items)

# Save splits for future reference/ground truth tracking
with open('train_set.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

with open('test_set.json', 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)

print(f"Dataset Split Complete!")
print(f"Train Set: {len(train_data)} items")
print(f"Test Set: {len(test_data)} items")

Dataset Split Complete!
Train Set: 16 items
Test Set: 4 items


In [6]:
import sys
sys.path.append('.') # Add kokoclone root to path for imports
from core.cloner import KokoClone
import os

# Initialize KokoClone (this handles downloading Kokoro weights automatically)
print("Initializing KokoClone...")
cloner = KokoClone()

# Create output directories
os.makedirs("evaluation_outputs/male", exist_ok=True)
os.makedirs("evaluation_outputs/female", exist_ok=True)

print("Starting generation on Unseen Test Set...")

for key, value in test_data.items():
    text = value["text"]
    item_id = value["id"]

    print(f"Processing {item_id}: {text[:30]}...")

    # 1. Generate Male Voice
    male_out_path = f"evaluation_outputs/male/{item_id}_male.wav"
    cloner.generate(
        text=text,
        lang="hi",
        reference_audio="ref_male.wav",
        output_path=male_out_path
    )

    # 2. Generate Female Voice
    female_out_path = f"evaluation_outputs/female/{item_id}_female.wav"
    cloner.generate(
        text=text,
        lang="hi",
        reference_audio="ref_female.wav",
        output_path=female_out_path
    )

print("Batch processing complete! Check the 'evaluation_outputs' folder.")

[2026-06-16 08:53:58,470] WARNING kanade_tokenizer: FlashAttention is not installed. Falling back to PyTorch SDPA implementation. There is no warranty that the model will work correctly.


Initializing KokoClone...
Initializing KokoClone on: CUDA
Loading Kanade model...


config.yaml:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/480M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/torchaudio/models/wavlm_base_plus.pth" to /root/.cache/torch/hub/checkpoints/wavlm_base_plus.pth


...

pytorch_model.bin:   0%|          | 0.00/54.4M [00:00<?, ?B/s]

Starting generation on Unseen Test Set...
Processing HIN_01: एक आम आदमी और औरत इमली के पेड़...


model/kokoro.onnx:   0%|          | 0.00/92.4M [00:00<?, ?B/s]

voice/voices-v1.0.bin:   0%|          | 0.00/28.2M [00:00<?, ?B/s]

Synthesizing text (HI)...
Applying Voice Clone...
[chunked_convert] VRAM budget: 13.11 GB (90% of 14.56 GB) → chunk size: 7.4s / 178,416 samples (RoPE ceiling: 7.4s)
[chunked_convert] Completed in 2.0s
Success! Saved: evaluation_outputs/male/HIN_01_male.wav
Synthesizing text (HI)...
Applying Voice Clone...
[chunked_convert] VRAM budget: 13.11 GB (90% of 14.56 GB) → chunk size: 7.4s / 178,416 samples (RoPE ceiling: 7.4s)
[chunked_convert] Completed in 0.2s
Success! Saved: evaluation_outputs/female/HIN_01_female.wav
Processing HIN_18: सेठ जी ने पचहत्तर प्रतिशत मुना...
Synthesizing text (HI)...
Applying Voice Clone...
[chunked_convert] VRAM budget: 13.11 GB (90% of 14.56 GB) → chunk size: 7.4s / 178,416 samples (RoPE ceiling: 7.4s)
[chunked_convert] Completed in 0.3s
Success! Saved: evaluation_outputs/male/HIN_18_male.wav
Synthesizing text (HI)...
Applying Voice Clone...
[chunked_convert] VRAM budget: 13.11 GB (90% of 14.56 GB) → chunk size: 7.4s / 178,416 samples (RoPE ceiling: 7.4s)
[ch

In [ ]:
!pip install speechbrain transformers jiwer torchaudio librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 48.1 MB/s eta 0:00:00


In [ ]:
!find /content -type d -name "evaluation_outputs*"

/content/kokoclone/evaluation_outputs


In [ ]:
import os
import json
import torch
import torchaudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from speechbrain.inference.speaker import EncoderClassifier
import jiwer
import numpy as np

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running evaluation on: {DEVICE}")

# Load the original dataset to get the ground truth text
with open('/content/hindi_evaluation_set.json', 'r', encoding='utf-8') as f:
    ground_truth_data = json.load(f)

# --- Initialize Models ---
print("Loading Whisper for ASR (Word Error Rate)...")
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(DEVICE)
# Force Hindi language for transcription
asr_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")

print("Loading SpeechBrain for Speaker Verification (Cosine Similarity)...")
# Using ECAPA-TDNN, a standard model for extracting speaker embeddings
speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="tmp_dir",
    run_opts={"device": DEVICE}
)

# --- Helper Functions ---
def transcribe_audio(audio_path):
    """Transcribes audio to Hindi text using Whisper."""
    waveform, sample_rate = torchaudio.load(audio_path)
    # Whisper expects 16kHz audio
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        predicted_ids = asr_model.generate(inputs.input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

def calculate_speaker_similarity(ref_path, gen_path):
    """Calculates Cosine Similarity between two audio files."""
    # Speechbrain's encode_batch handles loading and resampling internally
    with torch.no_grad():
        emb_ref = speaker_model.encode_batch(speaker_model.load_audio(ref_path)).squeeze()
        emb_gen = speaker_model.encode_batch(speaker_model.load_audio(gen_path)).squeeze()

    # Cosine similarity calculation
    cos = torch.nn.CosineSimilarity(dim=0)
    similarity = cos(emb_ref, emb_gen).item()
    return similarity

# --- Evaluation Loop ---
print("\nStarting Evaluation...")

results = {"male": [], "female": []}
genders = [("male", "ref_male.wav"), ("female", "ref_female.wav")]

for gender, ref_audio in genders:
    print(f"\n--- Evaluating {gender.capitalize()} Voices ---")
    folder_path = f"/content/kokoclone/evaluation_outputs/{gender}"

    if not os.path.exists(folder_path):
        print(f"Skipping {gender}: Folder not found.")
        continue

    for item_id, details in ground_truth_data.items():
        gen_audio_path = os.path.join(folder_path, f"{item_id}_{gender}.wav")
        ground_truth_text = details["text"]

        if not os.path.exists(gen_audio_path):
            continue

        # 1. Calculate WER
        transcription = transcribe_audio(gen_audio_path)
        # jiwer needs lists of strings
        wer_score = jiwer.wer(ground_truth_text, transcription)

        # 2. Calculate Speaker Similarity
        sim_score = calculate_speaker_similarity(ref_audio, gen_audio_path)

        results[gender].append({
            "id": item_id,
            "wer": wer_score,
            "similarity": sim_score
        })

        print(f"ID: {item_id} | Similarity: {sim_score:.3f} | WER: {wer_score:.3f}")

# --- Summary Statistics ---
for gender in ["male", "female"]:
    if results[gender]:
        avg_wer = np.mean([x["wer"] for x in results[gender]])
        avg_sim = np.mean([x["similarity"] for x in results[gender]])
        print(f"\n{gender.upper()} SUMMARY:")
        print(f"Average Cosine Similarity: {avg_sim:.3f}")
        print(f"Average Word Error Rate: {avg_wer:.3f}")

# Save results to disk
with open("evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
print("\nEvaluation complete. Detailed metrics saved to evaluation_metrics.json")

Running evaluation on: cpu
Loading Whisper for ASR (Word Error Rate)...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/kokoclone/kokoclone/tmp_dir/hyperparams.yaml'


Loading SpeechBrain for Speaker Verification (Cosine Similarity)...


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/kokoclone/kokoclone/tmp_dir/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/kokoclone/kokoclone/tmp_dir/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/kokoclone/kokoclone/tmp_dir/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/kokoclone/kokoclone/tmp_dir/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



Starting Evaluation...

--- Evaluating Male Voices ---

--- Evaluating Female Voices ---

Evaluation complete. Detailed metrics saved to evaluation_metrics.json


In [7]:
import os
import zipfile
from google.colab import files

# The exact absolute path you found
folder_path = '/content/kokoclone/evaluation_outputs'
zip_name = 'kokoclone_evaluation_outputs.zip'

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist.")
else:
    print(f"Zipping {folder_path}...")

    # Create the zip file
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(folder_path):
            for file in files_in_dir:
                file_path = os.path.join(root, file)
                # Keep the internal folder structure neat
                archive_name = os.path.relpath(file_path, os.path.dirname(folder_path))
                zipf.write(file_path, archive_name)

    print(f"Zip creation complete! Initiating download for {zip_name}...")

    # Trigger the browser download
    files.download(zip_name)

Zipping /content/kokoclone/evaluation_outputs...
Zip creation complete! Initiating download for kokoclone_evaluation_outputs.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>